# Cifrium Retention Intelligence
## D14 Early-Warning System for Student Churn

**North Star:** Course Completion Rate  
**Leading retention metric:** Module 1 → Module 2 Transition Rate  
**Operating constraint:** curators can contact only the highest-risk share of a cohort

### Executive result

The selected D14 model combines **media + training behavior** and reaches:

- ROC-AUC: **0.917**
- PR-AUC: **0.815**
- Precision@Top-20%: **85.4%**
- Recall@Top-20%: **55.9%**
- Lift@Top-20%: **2.79×**

The project is designed around a product decision: **who should receive retention attention first?**


# 1. Business problem

Cifrium loses **36% of students between Module 1 and Module 2**.

This makes early-course retention the highest-value intervention zone.

The prediction moment is fixed at **Day 14 of actual learning**, early enough for a curator intervention before the first-module retention outcome is fully realized.


In [ ]:
from pathlib import Path
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, precision_recall_curve

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src.features import (
    build_student_base,
    build_d14_media_features,
    build_d14_training_features,
)

DATA = Path(os.getenv("CIFRIUM_DATA_DIR", ROOT / "data"))
RESULTS = ROOT / "results"


# 2. Data and target definition

The target is the matured Module-1 status:

- `1 = churn`
- `0 = completed`

Final status is used **only as the label**. Final progress and whole-module outcome fields are not predictors.


In [ ]:
stats = pd.read_csv(DATA / "stats__module_1.csv")
groups = pd.read_csv(DATA / "groups.csv")
media = pd.read_csv(DATA / "wk_media_view_sessions.csv")
user_trainings = pd.read_csv(DATA / "user_trainings.csv")
trainings = pd.read_csv(DATA / "trainings.csv")

base = build_student_base(stats, groups)
print(f"Students with a valid learning start: {len(base):,}")
print(f"Overall churn rate: {base['churn'].mean():.1%}")


# 3. Leakage audit and D14 clock

A critical analytical issue is that **administrative enrollment is not the learning start**.

The median gap is about 26 days. D14 is therefore anchored to the first scheduled lesson of the student's parallel:

```text
stats.id параллели
        ↕
groups.group_template_id
        ↓
min(groups.starts_at)
        ↓
learning_start + 14 days
```

Every behavioral event must be timestamped at or before the D14 cutoff.


In [ ]:
lead = base["enrollment_lead_days"].dropna()
print("Median days between enrollment and learning start:", round(lead.median(), 1))

plt.figure(figsize=(8, 4))
plt.hist(lead.clip(-10, 80), bins=35)
plt.title("Enrollment date vs. actual learning start")
plt.xlabel("Days between enrollment and learning start")
plt.ylabel("Students")
plt.tight_layout()
plt.show()


## Training-specific leakage rule

A training may start before D14 and finish after D14.

Therefore:

- `started_at <= D14` → the start can be counted;
- `finished_at <= D14` → solved tasks / points can be used;
- `mark_saved_at <= D14` → mark can be used.

Outcome-like values are never taken from a later snapshot simply because the training itself started early.


# 4. D14 feature store


In [ ]:
media_df = build_d14_media_features(base, media)
training_df = build_d14_training_features(base, user_trainings, trainings)

training_cols = [c for c in training_df.columns if c != "user_id"]
df = media_df.merge(training_df[["user_id"] + training_cols], on="user_id", how="left")

print(f"Final modeling population: {len(df):,}")
print(f"Students with D14 training activity: {(df['tr_started_d14'] > 0).mean():.1%}")


The feature store captures two complementary dimensions:

### Media engagement
Frequency, active days, content breadth, watch depth, recency, inactivity gaps, and week-2 continuation.

### Training engagement
Training starts, active training days, safely completed activities, solved tasks, points available by D14, marks available by D14, and week-2 continuation.


# 5. Product EDA: what early churn looks like


In [ ]:
behavior = pd.read_csv(RESULTS / "behavioral_summary.csv")
behavior


In [ ]:
plot = behavior.set_index("metric")[["retained_mean", "churn_mean"]]
plot.plot(kind="barh", figsize=(9, 6))
plt.title("D14 behavior: retained vs. future churn")
plt.xlabel("Mean value")
plt.ylabel("")
plt.tight_layout()
plt.show()


The main pattern is a **loss of learning cadence**, not one isolated action.

Future churners return less often, consume fewer resources, show weaker week-2 continuation, and have much longer inactivity by D14.


# 6. Chronological validation

A random row split would overstate confidence because future cohorts may behave differently.

The headline evaluation uses later learning-start cohorts as the holdout.


In [ ]:
date_counts = df.groupby("learning_start").size().sort_index()
cumulative = date_counts.cumsum() / date_counts.sum()
split_date = cumulative[cumulative <= 0.80].index[-1]

train = df[df["learning_start"] <= split_date].copy()
test = df[df["learning_start"] > split_date].copy()

print(f"Train: {len(train):,} students | churn {train.churn.mean():.1%}")
print(f"Future holdout: {len(test):,} students | churn {test.churn.mean():.1%}")
print("Split date:", split_date)


# 7. Feature-source ablation

The question is not “how many tables can be joined?” but “which sources improve the decision?”


In [ ]:
ablation = pd.read_csv(RESULTS / "feature_source_ablation.csv")
ablation


In [ ]:
x = np.arange(len(ablation))
width = 0.35

plt.figure(figsize=(9, 4))
plt.bar(x - width/2, ablation["precision_at_20"], width, label="Precision@20%")
plt.bar(x + width/2, ablation["recall_at_20"], width, label="Recall@20%")
plt.xticks(x, ablation["feature_set"], rotation=20, ha="right")
plt.ylabel("Rate")
plt.title("Feature-source ablation at the operating Top-20% queue")
plt.legend()
plt.tight_layout()
plt.show()


**Decision:** select **media + training**.

The all-source model is marginally stronger on global ROC-AUC / PR-AUC, but media + training is materially stronger on the actual Top-20% intervention queue.


# 8. Final model

A regularized Logistic Regression is intentionally used as the operating model.

The goal is not algorithmic complexity. The goal is robust ranking under a capacity constraint.


In [ ]:
meta = [
    "enrollment_lead_days", "cohort_size_at_start",
    "start_weekday", "start_hour", "Уровень", "course_id"
]
media_features = [c for c in df.columns if c.startswith("media_")]
training_features = [c for c in df.columns if c.startswith("tr_")]
features = list(dict.fromkeys(meta + media_features + training_features))

categorical = ["start_weekday", "Уровень", "course_id"]
numeric = [c for c in features if c not in categorical]

prep = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical),
])

model = Pipeline([
    ("prep", prep),
    ("model", LogisticRegression(
        C=0.1,
        max_iter=5000,
        class_weight="balanced",
        random_state=42,
    )),
])

model.fit(train[features], train["churn"])
score = model.predict_proba(test[features])[:, 1]

def topk_metrics(y, score, share=.20):
    y = np.asarray(y)
    score = np.asarray(score)
    n = int(np.ceil(len(y) * share))
    idx = np.argsort(-score)[:n]
    precision = y[idx].mean()
    recall = y[idx].sum() / y.sum()
    lift = precision / y.mean()
    return precision, recall, lift

precision20, recall20, lift20 = topk_metrics(test["churn"], score)

print("ROC-AUC:", round(roc_auc_score(test["churn"], score), 3))
print("PR-AUC:", round(average_precision_score(test["churn"], score), 3))
print("Brier:", round(brier_score_loss(test["churn"], score), 3))
print("Precision@20%:", f"{precision20:.1%}")
print("Recall@20%:", f"{recall20:.1%}")
print("Lift@20%:", f"{lift20:.2f}x")


# 9. Precision–Recall view


In [ ]:
precision, recall, _ = precision_recall_curve(test["churn"], score)

plt.figure(figsize=(7, 5))
plt.plot(recall, precision)
plt.axhline(test["churn"].mean(), linestyle="--", label="Holdout churn baseline")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall curve on future cohorts")
plt.legend()
plt.tight_layout()
plt.show()


# 10. Temporal robustness


In [ ]:
temporal = pd.read_csv(RESULTS / "temporal_validation_extended.csv")
temporal


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(temporal["fold"], temporal["roc_auc"], marker="o", label="ROC-AUC")
plt.plot(temporal["fold"], temporal["pr_auc"], marker="o", label="PR-AUC")
plt.xticks(temporal["fold"])
plt.xlabel("Temporal fold")
plt.ylabel("Score")
plt.title("Expanding-window validation")
plt.legend()
plt.tight_layout()
plt.show()


The target prevalence changes sharply across time slices. This is not just an ML concern: cohort composition and retention conditions are changing.

Production monitoring must therefore include cohort churn rate, score distribution, Top-K quality, and drift.


# 11. Capacity analysis


In [ ]:
capacity = pd.read_csv(RESULTS / "capacity_curve.csv")
capacity


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(capacity["capacity"], capacity["precision"], marker="o", label="Precision")
plt.plot(capacity["capacity"], capacity["recall"], marker="o", label="Recall")
plt.xlabel("Share of cohort contacted")
plt.ylabel("Rate")
plt.title("Operational capacity trade-off")
plt.legend()
plt.tight_layout()
plt.show()


The operating threshold is chosen from **curator capacity and intervention economics**, not from a generic probability cutoff of 0.5.

The portfolio recommendation is **Top 20%**.


# 12. Risk segmentation and intervention

The model produces a ranked queue:

| Risk segment | Action |
|---|---|
| High | Personal curator outreach within 24h |
| Medium | Automated nudge + study-plan recommendation |
| Low | Standard learning journey |

Behavioral diagnostics should accompany the score so the curator sees whether the risk is associated with inactivity, weak week-2 continuation, low media engagement, or low training activity.


# 13. A/B test design

Prediction is not causal impact.

Among eligible high-risk D14 students:

- **Control:** business-as-usual
- **Treatment:** standardized curator outreach

**Primary metric:** M1 → M2 Transition Rate  
**Secondary metric:** Course Completion Rate

Guardrails: curator workload, intervention cost, complaints / opt-outs.

```text
Incremental retained students
=
Eligible contacted students
×
Measured causal uplift
```


# 14. Business impact logic

The ML model creates value only if three conditions hold:

1. risk ranking is better than random targeting;
2. the intervention creates positive causal uplift;
3. the uplift is worth the operational cost.

The project therefore separates **prediction quality** from **intervention effectiveness**.


# 15. Limitations and next steps

### Limitations
- training behavior covers fewer students than media behavior;
- churn prevalence shifts materially across cohorts;
- longer out-of-time validation is desirable;
- predictive associations are not causal drivers;
- additional raw event sources require a heavier scalable mapping pipeline.

### Next steps
1. run the high-risk intervention experiment;
2. monitor calibration and Top-K metrics by cohort;
3. add outcome economics to threshold selection;
4. evaluate additional timestamped task events when a scalable point-in-time join is available;
5. validate downstream impact on Course Completion Rate.
